[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/05_cuda_graphs_decode.ipynb)

# 5 · CUDA graphs for small-batch decoding
Weeks 14 and 25. At batch 1 a decode step is dozens of tiny kernels; CPU launch overhead (~5–10 µs each) can dominate. A CUDA graph
records the whole step once and replays it with one launch. It needs **static shapes and addresses** — hence the preallocated KV
cache in `gpu/decode.py`, written in place at position `pos`.

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
from gpu.decode import TinyDecoder, capture_step
torch.manual_seed(0)
m = TinyDecoder(vocab=32000, d=512, n_layers=8, n_heads=8, max_len=512).cuda().eval()
B, steps = 1, 200
def eager_decode():
    cache = m.new_cache(B, "cuda"); tok = torch.zeros(B, dtype=torch.long, device="cuda")
    with torch.inference_mode():
        for t in range(steps):
            tok = m.step(tok, torch.tensor(t, device="cuda"), cache).argmax(-1)
    return tok
g, s_tok, s_pos, cache, s_logits = capture_step(m, B)
def graph_decode():
    cache.zero_(); s_tok.zero_()
    for t in range(steps):
        s_pos.fill_(t); g.replay(); s_tok.copy_(s_logits.argmax(-1))
    return s_tok
assert torch.equal(eager_decode(), graph_decode()), "graph replay must give identical tokens"
import time
for name, f in [("eager", eager_decode), ("cuda graph", graph_decode)]:
    f(); torch.cuda.synchronize(); t0 = time.perf_counter(); f(); torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    print(f"{name:>10}: {steps / dt:7.0f} tokens/s  ({dt / steps * 1e3:.2f} ms/token)")

## Your turn
1. Repeat with batch 32 and with `d=2048`. When do graphs stop mattering, and why (compute- vs launch-bound)?
2. `torch.compile(mode="reduce-overhead")` uses CUDA graphs automatically — compare it with the manual capture.
3. Why must the sampling step (argmax/top-p) also be inside the graph for best results? Try moving it in.